
# Example 3 — Nonlinear cylinder target in \(d=2\): variance collapse toward a compact blob

This notebook implements **Algorithm 2** / the **nonlinear cylinder terminal condition** in a setup
whose effect is easy to see and fairly stable across random seeds.

A direct Euclidean variance observable is not periodic on the flat torus \([0,1)^2\), so instead we use the smooth
torus-friendly proxy
\[
\phi_c(x,y)=\frac12\Big(\cos\!\big(2\pi(x-c_x)\big)+\cos\!\big(2\pi(y-c_y)\big)\Big),
\]
centered at \(c=(c_x,c_y)\).

Why this is a good "small-variance" proxy:
- \(\phi_c \le 1\), with equality only at the target center \(c\)
- for small periodic displacements \(\delta=(\delta_x,\delta_y)\),
  \[
  \phi_c(c+\delta)\approx 1-\pi^2(\delta_x^2+\delta_y^2),
  \]
  so making the weighted average \(\mu(\phi_c)\) large is, locally, the same as making the second moment around \(c\) small
- the conditioning is still **collective / unlabelled**: only the global score
  \[
  \mu(\phi_c)=\sum_{i=1}^n s_i \phi_c(x_i)
  \]
  matters, not which particle ends up closest to the center

The experiment below starts from a **large square** of particles and steers them toward a compact blob near the center of the torus.

We use two diagnostics:
1. the collective cosine score \(\mu_t(\phi_c)\)
2. the weighted periodic RMS radius
   \[
   r_t=\Big(\sum_{i=1}^n s_i\, d_{\mathbb T^2}(x_i(t),c)^2\Big)^{1/2},
   \]
   which is the most direct variance-like quantity for this setup.


In [ ]:
import numpy as np
import sys
from pathlib import Path
from numpy.polynomial.hermite import hermgauss


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.wasserstein_conditioning_algorithms import (
    shortest_periodic_displacement,
    simulate_nonlinear_cylinder_quadrature_em,
)

np.set_printoptions(precision=3, suppress=True)


In [ ]:
import plotly.graph_objects as go

from notebooks.support import (
    center_trace,
    circle_trace,
    configure_plotly,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)

configure_plotly()


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=6):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=18,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        particle_name="particles",
        mass_format=".3f",
        time_formatter=lambda t, h: f"time = {t:.3f}",
        slider_label_formatter=lambda t, h: f"{t:.3f}",
        currentvalue_prefix="time = ",
        width=800,
        height=700,
        play_frame_duration=110,
    )


In [ ]:

def cosine_blob_observable(center=(0.5, 0.5)):
    center = np.asarray(center, dtype=float)

    def phi(points):
        pts = np.asarray(points, dtype=float)
        return 0.5 * (
            np.cos(2.0 * np.pi * (pts[..., 0] - center[0]))
            + np.cos(2.0 * np.pi * (pts[..., 1] - center[1]))
        )

    return phi

def gauss_hermite_cylinder_quadrature(lambda_, order):
    nodes, weights = hermgauss(order)
    eta = np.sqrt(2.0 * lambda_) * nodes[:, None]
    quad_weights = weights / np.sqrt(np.pi)
    return eta, quad_weights

def weighted_observable_score(positions, masses, observable):
    return np.array([np.sum(masses * observable(pos)) for pos in positions], dtype=float)

def weighted_periodic_rms_radius(positions, masses, center):
    center = np.asarray(center, dtype=float)
    displacements = shortest_periodic_displacement(positions, center[None, None, :])
    squared_radii = np.sum(displacements ** 2, axis=-1)
    return np.sqrt(squared_radii @ masses)

def simulate_free_diffusion(masses, initial_positions, horizon, step_size, seed):
    rng = np.random.default_rng(seed)
    masses = np.asarray(masses, dtype=float)
    positions0 = np.mod(np.asarray(initial_positions, dtype=float), 1.0)

    m_steps = int(round(horizon / step_size))
    times = np.linspace(0.0, horizon, m_steps + 1, dtype=float)
    positions = np.empty((m_steps + 1, len(masses), positions0.shape[1]), dtype=float)
    positions[0] = positions0

    state = positions0.copy()
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    for m in range(m_steps):
        state = np.mod(state + noise_scale * rng.normal(size=state.shape), 1.0)
        positions[m + 1] = state

    return times, positions

# --- masses and geometry ---
masses = np.array([0.40, 0.30, 0.20, 0.10], dtype=float)
masses = masses / masses.sum()

target_center = np.array([0.50, 0.50], dtype=float)
initial_positions = np.array([
    [0.15, 0.15],
    [0.15, 0.85],
    [0.85, 0.85],
    [0.85, 0.15],
], dtype=float)

square_outline = initial_positions.copy()
compact_guide_radius = 0.20

observable = cosine_blob_observable(target_center)

# Algorithm 2 parameters:
target_score = 0.80
lambda_ = 20.0
horizon = 0.03
step_size = horizon / 100
quadrature_order = 21
grid_shape = 40
seed = 0

quadrature_nodes, quadrature_weights = gauss_hermite_cylinder_quadrature(lambda_, quadrature_order)

def run_conditioned_simulation(seed, *, store_drifts=True):
    rng = np.random.default_rng(seed)
    return simulate_nonlinear_cylinder_quadrature_em(
        masses=masses,
        observables=[observable],
        target_vector=np.array([target_score], dtype=float),
        lambda_=lambda_,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        quadrature_nodes=quadrature_nodes,
        quadrature_weights=quadrature_weights,
        grid_shape=grid_shape,
        rng=rng,
        store_drifts=store_drifts,
    )

initial_score = float(np.sum(masses * observable(initial_positions)))
initial_rms_radius = float(weighted_periodic_rms_radius(initial_positions[None, :, :], masses, target_center)[0])

print("initial weighted cosine score:", initial_score)
print("initial weighted periodic RMS radius:", initial_rms_radius)
print("target score a:", target_score)


In [ ]:

sim = run_conditioned_simulation(seed, store_drifts=True)
free_times, free_positions = simulate_free_diffusion(masses, initial_positions, horizon, step_size, seed)

score = weighted_observable_score(sim.positions, sim.masses, observable)
rms_radius = weighted_periodic_rms_radius(sim.positions, sim.masses, target_center)
free_rms_radius = weighted_periodic_rms_radius(free_positions, masses, target_center)

print("positions array shape:", sim.positions.shape)
print("final time:", float(sim.times[-1]))
print("final conditioned score:", float(score[-1]))
print("final conditioned RMS radius:", float(rms_radius[-1]))
print("final free-diffusion RMS radius:", float(free_rms_radius[-1]))


In [ ]:

static_traces = [
    line_trace(square_outline, name="initial square guide", color="rgba(30, 144, 255, 0.50)", dash="dash", close=True),
    circle_trace(target_center, radius=compact_guide_radius, name="compact guide (visual only)", color="rgba(220, 20, 60, 0.55)"),
    center_trace(target_center[None, :], name="target center", color="rgba(220, 20, 60, 0.8)", symbol="x", size=12, showlegend=False),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title="Example 3: nonlinear cylinder target (variance-collapse proxy)",
    static_traces=static_traces,
    marker_size=30,
)
fig.show()



### Free-diffusion baseline

The next animation uses the **same initial condition** and **same seed**, but with **no conditioning drift**.
This makes the variance-collapse effect easier to compare visually.


In [ ]:

free_fig = make_particle_animation(
    positions=free_positions,
    times=free_times,
    masses=masses,
    title="Free diffusion baseline (same initial state and same seed)",
    static_traces=static_traces,
    marker_size=30,
)
free_fig.show()


In [ ]:

score_fig = go.Figure()
score_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=score,
        mode="lines",
        name="conditioned weighted cosine score",
    )
)
score_fig.add_hline(
    y=target_score,
    line_dash="dash",
    annotation_text="target a",
    annotation_position="top left",
)
score_fig.update_layout(
    title="Collective observable over time",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted cosine score",
)
score_fig.show()

radius_fig = go.Figure()
radius_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=rms_radius,
        mode="lines",
        name="conditioned RMS radius",
    )
)
radius_fig.add_trace(
    go.Scatter(
        x=free_times,
        y=free_rms_radius,
        mode="lines",
        name="free RMS radius",
        opacity=0.8,
    )
)
radius_fig.add_hline(
    y=initial_rms_radius,
    line_dash="dash",
    annotation_text="initial RMS radius",
    annotation_position="top right",
)
radius_fig.update_layout(
    title="Variance-like diagnostic: weighted periodic RMS radius",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted periodic RMS radius",
)
radius_fig.show()

print("initial weighted cosine score:", float(score[0]))
print("final weighted cosine score:", float(score[-1]))
print("target score a:", float(target_score))
print()
print("initial RMS radius:", float(rms_radius[0]))
print("final conditioned RMS radius:", float(rms_radius[-1]))
print("final free RMS radius:", float(free_rms_radius[-1]))
print("conditioned radius reduction factor:", float(rms_radius[-1] / rms_radius[0]))
print("free radius reduction factor:", float(free_rms_radius[-1] / free_rms_radius[0]))



### Seed robustness check

To check stability, the next cell reruns the **same** setup for several seeds and compares the final
weighted periodic RMS radius against the corresponding **free diffusion** baseline.

For this example the conditioned runs are consistently more concentrated, and also less variable,
than the free runs.


In [ ]:

robustness_seeds = list(range(10))
conditioned_final_radii = []
free_final_radii = []

for s in robustness_seeds:
    sim_s = run_conditioned_simulation(s, store_drifts=False)
    conditioned_final_radii.append(float(weighted_periodic_rms_radius(sim_s.positions, masses, target_center)[-1]))

    _, free_positions_s = simulate_free_diffusion(masses, initial_positions, horizon, step_size, s)
    free_final_radii.append(float(weighted_periodic_rms_radius(free_positions_s, masses, target_center)[-1]))

conditioned_final_radii = np.array(conditioned_final_radii, dtype=float)
free_final_radii = np.array(free_final_radii, dtype=float)

robust_fig = go.Figure()
robust_fig.add_trace(
    go.Scatter(
        x=robustness_seeds,
        y=conditioned_final_radii,
        mode="lines+markers",
        name="conditioned final RMS radius",
    )
)
robust_fig.add_trace(
    go.Scatter(
        x=robustness_seeds,
        y=free_final_radii,
        mode="lines+markers",
        name="free final RMS radius",
        opacity=0.8,
    )
)
robust_fig.add_hline(
    y=float(conditioned_final_radii.mean()),
    line_dash="dash",
    annotation_text="conditioned mean",
    annotation_position="bottom right",
)
robust_fig.add_hline(
    y=float(free_final_radii.mean()),
    line_dash="dot",
    annotation_text="free mean",
    annotation_position="top right",
)
robust_fig.update_layout(
    title="Seed robustness: final RMS radius across several runs",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="seed",
    yaxis_title="final weighted periodic RMS radius",
)
robust_fig.show()

print("conditioned final RMS radii by seed:")
for s, value in zip(robustness_seeds, conditioned_final_radii):
    print(f"  seed {s}: {value:.6f}")

print()
print("free-diffusion final RMS radii by seed:")
for s, value in zip(robustness_seeds, free_final_radii):
    print(f"  seed {s}: {value:.6f}")

print()
print("conditioned mean final RMS radius:", float(conditioned_final_radii.mean()))
print("conditioned std of final RMS radius:", float(conditioned_final_radii.std()))
print("free mean final RMS radius:", float(free_final_radii.mean()))
print("free std of final RMS radius:", float(free_final_radii.std()))
print("mean conditioned / free ratio:", float(conditioned_final_radii.mean() / free_final_radii.mean()))
